## Interpret signal

In [117]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from scipy.signal import medfilt

# Load data
df = pd.read_csv("../LDR_5.1_data.csv")

time = df["time_s"].to_numpy()
adc = df["ADC"].to_numpy()

# Apply median filter
filtered = medfilt(adc, kernel_size=51)

# Plot raw and filtered signal
plt.figure(figsize=(15, 5))

plt.plot(time, adc, alpha=0.35, label="Raw signal")
plt.plot(time, filtered, linewidth=1.5, label="Filtered signal")

plt.xlabel("Time [s]")
plt.ylabel("ADC value")
plt.title("Light sensor - Morse code signal")
plt.legend()
plt.grid()
plt.tight_layout()
plt.show()


FileNotFoundError: [Errno 2] No such file or directory: '../LDR_5.1_data.csv'

## Convert the signal to morse code

In [ ]:
# Parameters
THRESHOLD = 600
FS = 5000

UNIT_TIME = 0.050
SAMPLES_PER_UNIT = 250

# Binary classification
binary = filtered >= THRESHOLD

# Find transitions
transitions = np.diff(binary.astype(int))
change_indices = np.where(transitions != 0)[0] + 1

# Construct intervals
boundaries = np.concatenate(([0], change_indices, [len(binary)]))

runs = []

for start, end in zip(boundaries[:-1], boundaries[1:]):
    state = binary[start]
    duration = end - start

    runs.append({
        "state": "ON" if state else "OFF",
        "start_sample": start,
        "end_sample": end,
        "duration_samples": duration,
        "duration_ms": duration / FS * 1000,
        "units": duration / SAMPLES_PER_UNIT
    })

runs_df = pd.DataFrame(runs)
runs = np.array(runs)

runs_df.to_csv("runsdftest")

print(runs_df.head(30).to_string(index=False))

state  start_sample  end_sample  duration_samples  duration_ms  units
  OFF             0         209               209         41.8  0.836
   ON           209         486               277         55.4  1.108
  OFF           486        1210               724        144.8  2.896
   ON          1210        1988               778        155.6  3.112
  OFF          1988        2209               221         44.2  0.884
   ON          2209        2989               780        156.0  3.120
  OFF          2989        3209               220         44.0  0.880
   ON          3209        3988               779        155.8  3.116
  OFF          3988        4710               722        144.4  2.888
   ON          4710        5488               778        155.6  3.112
  OFF          5488        5709               221         44.2  0.884
   ON          5709        6484               775        155.0  3.100
  OFF          6484        7210               726        145.2  2.904
   ON          7210 

In [ ]:
df_off = runs_df[runs_df["state"] == "OFF"]
df_off = df_off.sort_values(by="duration_ms", ascending=False)
df_off.head()

,state,start_sample,end_sample,duration_samples,duration_ms,units
864,OFF,430988,433455,2467,493.4,9.868
46,OFF,21984,23711,1727,345.4,6.908
186,OFF,94485,96211,1726,345.2,6.904
116,OFF,57485,59211,1726,345.2,6.904
462,OFF,229984,231710,1726,345.2,6.904


In [ ]:
runs = runs[:864]

In [ ]:

# Morse code dictionary
MORSE = {
    ".-": "A", 
    "-...": "B", 
    "-.-.": "C",
    "-..": "D", 
    ".": "E", 
    "..-.": "F",
    "--.": "G", 
    "....": "H", 
    "..": "I",
    ".---": "J", 
    "-.-": "K", 
    ".-..": "L",
    "--": "M",
    "-.": "N",
    "---": "O",
    ".--.": "P", 
    "--.-": "Q", 
    ".-.": "R",
    "...": "S", 
    "-": "T", 
    "..-": "U",
    "...-": "V", 
    ".--": "W", 
    "-..-": "X",
    "-.--": "Y", 
    "--..": "Z",

    # Numbers
    "-----": "0", 
    ".----": "1", 
    "..---": "2",
    "...--": "3", 
    "....-": "4", 
    ".....": "5",
    "-....": "6", 
    "--...": "7", 
    "---..": "8",
    "----.": "9",

    # Special chars
    "--..--": ",",
    ".-.-.-": ".",
    ".----.": "'",
}


In [ ]:
UNIT_TIME_MS = 50

# Timing thresholds
LONG_PULSE_THRESHOLD = 125 # signal on for more than 100ms 

NEW_LETTER_THRESHOLD = 120 # signal off for more than 100ms is a new character 
NEW_WORD_THRESHOLD   = 200 # signal off for more than 400ms is a new word

letters = []
current_letter = ""

# Loop over message data
for run in runs:
    duration = run["duration_ms"]

    # If the signal is high it is either a dot or a dash
    if run["state"] == "ON":

        # Short pulse
        if duration < LONG_PULSE_THRESHOLD:
            current_letter += "."

        # Long pulse  
        else:
            current_letter += "-"

    # Signal is low
    else:
        # 
        if NEW_LETTER_THRESHOLD <= duration :
            
            letters.append(current_letter)
            current_letter = ""

            if duration >= NEW_WORD_THRESHOLD:
                letters.append(" ")

# Handle final letter
if current_letter:
    letters.append(current_letter)

# Decode
decoded = []

for letter in letters:
    if letter == " ":
        decoded.append(" ")
    else:
        decoded.append(MORSE.get(letter, "?"))

message = "".join(decoded)

print(message)

EOMEWHERE IN ME IS A CURIOSITY SENSOR. I WANT TO KNOW WHAT'S OVER THE NEXT HILL. YOU KNOW, PEOPLE CAN LIVE LONGER WITHOUT FOOD THAN WITHOUT INFORMATION. WITHOUT INFORMATION, YOU'D GO CRAZY.
